
## Data Curation Built-in Functions:

- Data curation is the `process of cleaning, standardizing, validating, and transforming raw data into reliable data`.
- that is used for `analytics and reporting`.


**In real-time data engineering, raw data may contain:**
- Missing values.
- Extra spaces.
- Incorrect data types.
- Inconsistent date formats.
- Incomplete names or addresses.
- Invalid or inconsistent values.

PySpark provides `built-in functions` to handle these problems.

## Example: Raw employee data

In [0]:
from pyspark.sql import functions as F

employee_df = spark.createDataFrame([
    (101, "  Arun  ", "50000", "2026-01-15", None),
    (102, " Bala ", None, "2026-02-20", "IT"),
    (103, None, "45000", "2026-03-10", "HR"),
    (104, " Kumar ", "60000", "2026-04-05", "IT")
], ["EmployeeID", "Name", "Salary", "HireDate", "Department"])

employee_df.show(truncate=False)


## A. coalesce():

**coalesce()** is used to return a `first non-null value from a list of columns or expressions`.

**Example:**

If an employee's department is missing, assign "Not Assigned".

In [0]:
employee_df.withColumn(
    "Department",
    F.coalesce(F.col("Department"), F.lit("Not Assigned"))
).show()


## B. when() and otherwise():

- **when()** is used to `applies a value when a condition is true`. 
- **otherwise()** provides a `fallback when the condition is false or no condition matches`.


**Example:**
Classify employees according to salary.

In [0]:
employee_df.withColumn(
    "SalaryCategory",
    F.when(F.col("Salary").cast("int") >= 50000, "High")
     .when(F.col("Salary").cast("int") >= 45000, "Medium")
     .otherwise("Low or Missing")
).show()


## C. trim():

**trim()** is used to `removes leading and trailing spaces` from a string.

In [0]:
employee_df.withColumn(
    "CleanName",
    F.trim(F.col("Name"))
).show()


## D. isNull():

**isNull()** is used to `checks the column contains a null value` and `returns a Boolean expression`.

In [0]:
employee_df.filter(
    F.col("Name").isNull()
).show()


## To replace missing names:

In [0]:
employee_df.withColumn(
    "Name",
    F.when(F.col("Name").isNull(), "Unknown")
     .otherwise(F.col("Name"))
).show()


## E. lit():

**lit()** is used to `creates a literal column or expression from a constant value`.

In [0]:
employee_df.withColumn(
    "Country",
    F.lit("India")
).show()


**Real-time use: Add source-system labels, country codes, processing flags, or pipeline metadat**

In [0]:
# Step 2: Add pipeline metadata using lit()
employee_df.withColumn(
    "SourceSystem", F.lit("HR_System")
).withColumn(
    "CountryCode", F.lit("IN")
).withColumn(
    "ProcessingFlag", F.lit("PROCESSED")
).withColumn(
    "PipelineMetadata", F.lit("HR_ETL_Pipeline")
).show(truncate=False)


## F. cast():

**cast()** is used to `converts a column or expression into a specified data type`.


**Real-time example:**

**A CSV file may contain salary as a string. Convert it to a numeric type for calculations.**

In [0]:
clean_df = employee_df.withColumn(
    "Salary",
    F.col("Salary").cast("double")
)

clean_df.printSchema()


**To calculate average salary:**

In [0]:
clean_df.agg(
    F.avg("Salary").alias("AverageSalary")
).show()


**Real-time use: Convert string-formatted amounts, dates, and numeric identifiers into appropriate data types.**


## G. substr():
**substr()** is used to `extracts a substring from a string using a starting position and a length`.

In [0]:
employee_df.withColumn(
    "EmployeePrefix",
    F.col("EmployeeID").cast("string").substr(1, 2)
).show()


## H. to_date():
**to_date()** is used to `converts a date-formatted string or timestamp expression into a Spark DateType column`.

In [0]:
date_df = employee_df.withColumn(
    "HireDate",
    F.to_date(F.col("HireDate"), "yyyy-MM-dd")
)

date_df.printSchema()


**To filter employees hired after February 1, 2026:**

In [0]:
date_df.filter(
    F.col("HireDate") > F.lit("2026-02-01").cast("date")
).show()


## I. concat():

**concat()** is used to `combines multiple columns or expressions into one value`.

In [0]:
employee_df.withColumn(
    "EmployeeDetails",
    F.concat(
        F.col("EmployeeID").cast("string"),
        F.lit(" - "),
        F.trim(F.col("Name"))
    )
).show()


### Data Curation Functions:

| Function | Definition | Typical Use |
|---|---|---|
| `coalesce()` | Returns the first non-null expression from the given expressions. | Replace missing values with fallback values. |
| `when()` | Evaluates a condition and returns a specified value when the condition is true. | Apply conditional transformations. |
| `otherwise()` | Defines the default value when no preceding `when()` condition matches. | Assign default classifications. |
| `trim()` | Removes leading and trailing spaces from a string. | Clean unwanted spaces in text columns. |
| `isNull()` | Checks whether a column value is null. | Identify incomplete records. |
| `lit()` | Creates a literal value as a Spark column expression. | Add constant values to a DataFrame. |
| `cast()` | Converts a column or expression to a specified data type. | Correct or standardize data types. |
| `substr()` | Extracts a substring from a string column. | Extract parts of identifiers or codes. |
| `to_date()` | Converts a column or expression into a date value, optionally using a specified format. | Standardize date columns. |
| `concat()` | Combines two or more column expressions into a single string expression. | Build full names, composite keys, or formatted strings. |